# Is it legit to interpolate q25 / q75 from the calibrated q10 / q50 / q90?

We ship a 7-point predictive CDF per cell (quantile levels **0.01, 0.05, 0.10, 0.50,
0.90, 0.95, 0.99**), each non-median head CQR-shifted to its nominal one-sided coverage
(`make_debias_tables.compute_level_shifts`). q25 and q75 are **not** trained. This notebook
answers one question: if the client interpolates q25/q75 from the calibrated
q10/q50/q90 it already has, what empirical coverage do those interpolated quantiles get?

**Target:** a well-calibrated q25 has `P(bias <= q25) = 0.25`; q75 has `0.75`.

**Honest evaluation:** we split the held-out **test** blocks into two disjoint halves —
*calibration* blocks (fit the global CQR shifts, exactly like production) and *evaluation*
blocks (measure coverage). Neither the CatBoost fit nor the CQR shifts ever saw the eval
blocks, so the interpolated-quantile coverage there is an out-of-sample read.

Two interpolation schemes are compared:
- **probit** — linear in `Φ⁻¹(τ)` between the bracketing anchors (== a split-normal that
  respects the measured lower/upper skew). Recommended.
- **linear-τ** — naive linear in `τ` (the strawman).

In [1]:
import numpy as np, pandas as pd
from scipy.stats import norm
from catboost import CatBoostRegressor
import train_quantile_debias as tqd
import make_debias_tables as mdt   # reuse the PRODUCTION per-level CQR shift + isotonic

pd.set_option("display.width", 140, "display.float_format", lambda v: f"{v:.4f}")

Q = tqd.QUANTILES                    # [.01 .05 .10 .50 .90 .95 .99]
I10, I50, I90 = Q.index(0.10), Q.index(0.50), Q.index(0.90)
TAG = "qn8727_s0"
print("quantile levels:", Q, "| using anchors idx", I10, I50, I90)

quantile levels: [0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99] | using anchors idx 2 3 4


In [2]:
# --- load the cached post-split processed frame (add_features + split already baked) ---
frame = pd.read_feather("data/pipeline_frame_full.feather")
print(f"{len(frame):,} rows, {frame.key.nunique()} cells, "
      f"{frame.date.min().date()}..{frame.date.max().date()}")

# reconstruct the monthly block id exactly as tqd.split does, so we can carve the
# TEST blocks into disjoint calibration vs evaluation halves.
origin = frame.date.min().normalize()
block_id = ((frame.date.dt.year - origin.year) * 12
            + (frame.date.dt.month - origin.month))
frame["block"] = block_id
test_blocks = np.sort(frame.loc[frame.role == "test", "block"].unique())
calib_blocks = set(test_blocks[0::2])      # alternate test blocks -> calib / eval
eval_blocks  = set(test_blocks[1::2])
print(f"test blocks: {list(test_blocks)}")
print(f"  calib -> {sorted(calib_blocks)}")
print(f"  eval  -> {sorted(eval_blocks)}")

7,179,885 rows, 8727 cells, 2024-03-01..2026-06-01


test blocks: [np.int32(0), np.int32(5), np.int32(10), np.int32(15), np.int32(20), np.int32(25)]
  calib -> [np.int32(0), np.int32(10), np.int32(20)]
  eval  -> [np.int32(5), np.int32(15), np.int32(25)]


In [3]:
# --- interpolation schemes: q25 lives in the lower segment [q10,q50], q75 in [q50,q90] ---
z = {lvl: float(norm.ppf(lvl)) for lvl in (0.10, 0.25, 0.50, 0.75, 0.90)}

def interp_probit(q10, q50, q90):
    "linear in probit space between the bracketing anchors (== segment split-normal)"
    w25 = (z[0.25] - z[0.10]) / (z[0.50] - z[0.10])
    w75 = (z[0.75] - z[0.50]) / (z[0.90] - z[0.50])
    return q10 + w25 * (q50 - q10), q50 + w75 * (q90 - q50)

def interp_linear_tau(q10, q50, q90):
    "naive linear in tau"
    w25 = (0.25 - 0.10) / (0.50 - 0.10)     # 0.375
    w75 = (0.75 - 0.50) / (0.90 - 0.50)     # 0.625
    return q10 + w25 * (q50 - q10), q50 + w75 * (q90 - q50)

print("probit weights :", interp_probit(0., 1., 2.))     # (w25 offset from q10, ...)
print("lin-tau weights:", interp_linear_tau(0., 1., 2.))

probit weights : (0.4736928514386742, 1.526307148561326)
lin-tau weights: (0.37499999999999994, 1.625)


In [4]:
def calibrated_trio(model, rows, name):
    "sorted 7-head preds -> global one-sided CQR shifts fit on CALIB -> apply to EVAL, "
    "return monotone (q10,q50,q90) on eval rows + the eval truth."
    feat = tqd.feature_cols(name, with_cell=True, with_cross=False)
    X = rows[feat].astype({"key": str, "fc_version": str})
    preds = np.sort(np.asarray(model.predict(X)), axis=1)     # (n,7), production sorts
    y = rows[f"bias_{name}"].to_numpy()
    is_calib = rows.block.isin(calib_blocks).to_numpy()
    is_eval  = rows.block.isin(eval_blocks).to_numpy()

    # global per-level CQR shift, fit on CALIB only (mirror compute_level_shifts)
    shifts = np.zeros(len(Q))
    for i, lvl in enumerate(Q):
        if i != I50:
            shifts[i] = mdt.per_level_shift(y[is_calib] - preds[is_calib, i], lvl)

    adj = np.round(preds[is_eval] + shifts, 2)
    adj = mdt.pin_median_isotonic(adj)          # re-isotonize around pinned median
    return adj[:, I10], adj[:, I50], adj[:, I90], y[is_eval], shifts

def coverage(qhat, y):
    return float(np.mean(y <= qhat))

In [5]:
records = []
trio_cache = {}
for var in tqd.VARS:
    name = var["name"]
    model = CatBoostRegressor(); model.load_model(str(tqd.MODELS / f"M3_base_{name}_{TAG}.cbm"))
    rows = frame[(frame.role == "test") & frame[f"bias_{name}"].notna()]
    q10, q50, q90, y_eval, shifts = calibrated_trio(model, rows, name)
    trio_cache[name] = (rows, q10, q50, q90, y_eval)

    # anchor sanity (CQR out-of-sample) + interpolated coverage
    q25_p, q75_p = interp_probit(q10, q50, q90)
    q25_l, q75_l = interp_linear_tau(q10, q50, q90)
    records.append(dict(
        var=name, n_eval=len(y_eval),
        cov_q10=coverage(q10, y_eval), cov_q50=coverage(q50, y_eval), cov_q90=coverage(q90, y_eval),
        q25_probit=coverage(q25_p, y_eval), q25_linTau=coverage(q25_l, y_eval),
        q75_probit=coverage(q75_p, y_eval), q75_linTau=coverage(q75_l, y_eval)))
    del model

results = pd.DataFrame(records).set_index("var")
print("Empirical coverage on disjoint EVAL blocks  (targets: q10=.10 q25=.25 q50=.50 q75=.75 q90=.90)\n")
results

Empirical coverage on disjoint EVAL blocks  (targets: q10=.10 q25=.25 q50=.50 q75=.75 q90=.90)



,n_eval,cov_q10,cov_q50,cov_q90,q25_probit,q25_linTau,q75_probit,q75_linTau
var,,,,,,,,
tmax,794157,0.0934,0.4936,0.8975,0.2327,0.1956,0.7551,0.7928
tmin,794157,0.0933,0.5043,0.9034,0.2399,0.2014,0.7631,0.8002
precip,794157,0.5717,0.7224,0.9153,0.6427,0.6291,0.8525,0.8695
wind,794157,0.1017,0.4976,0.8997,0.2478,0.2105,0.7537,0.7915


In [6]:
# error vs nominal, in PERCENTAGE POINTS — the number that answers 'how legit'
pp = pd.DataFrame({
    "q10_anchor": (results.cov_q10 - .10) * 100,
    "q50_anchor": (results.cov_q50 - .50) * 100,
    "q90_anchor": (results.cov_q90 - .90) * 100,
    "q25 PROBIT": (results.q25_probit - .25) * 100,
    "q25 linTau": (results.q25_linTau - .25) * 100,
    "q75 PROBIT": (results.q75_probit - .75) * 100,
    "q75 linTau": (results.q75_linTau - .75) * 100,
}).round(2)
print("Coverage error vs nominal (percentage points; + = over-covers)\n")
pp

Coverage error vs nominal (percentage points; + = over-covers)



,q10_anchor,q50_anchor,q90_anchor,q25 PROBIT,q25 linTau,q75 PROBIT,q75 linTau
var,,,,,,,
tmax,-0.6600,-0.6400,-0.2500,-1.7300,-5.4400,0.5100,4.2800
tmin,-0.6700,0.4300,0.3400,-1.0100,-4.8600,1.3100,5.0200
precip,47.1700,22.2400,1.5300,39.2700,37.9100,10.2500,11.9500
wind,0.1700,-0.2400,-0.0300,-0.2200,-3.9500,0.3700,4.1500


In [7]:
# robustness: does interpolated q25/q75 hold ACROSS forecast regimes, or only marginally?
# bucket eval rows by the forecast decile label fdec (low / mid / high), probit scheme.
cond = []
for name, (rows, q10, q50, q90, y_eval) in trio_cache.items():
    q25_p, q75_p = interp_probit(q10, q50, q90)
    fdec_eval = rows.loc[rows.block.isin(eval_blocks), f"fdec_{name}"].to_numpy()
    for bucket in ("low", "mid", "high"):
        m = fdec_eval == bucket
        if m.sum() == 0: continue
        cond.append(dict(var=name, fdec=bucket, n=int(m.sum()),
                         q25=coverage(q25_p[m], y_eval[m]),
                         q75=coverage(q75_p[m], y_eval[m])))
cond_cov = pd.DataFrame(cond)
print("Conditional coverage by forecast decile (probit) — target q25=.25, q75=.75\n")
cond_cov.pivot(index="var", columns="fdec", values=["q25", "q75"])[
    [("q25","low"),("q25","mid"),("q25","high"),("q75","low"),("q75","mid"),("q75","high")]]

Conditional coverage by forecast decile (probit) — target q25=.25, q75=.75



q25                  q75              
fdec      low    mid   high    low    mid   high
var                                             
precip 0.4324 0.6768 0.3272 0.7250 0.8586 0.8125
tmax   0.2001 0.2323 0.2749 0.7031 0.7585 0.7807
tmin   0.1717 0.2426 0.2976 0.6878 0.7685 0.7976
wind   0.2377 0.2465 0.2693 0.7503 0.7544 0.7513

In [8]:
# one-line verdict per variable: worst-case interpolated-quantile miss (probit), in pp
worst = pd.concat([
    (results.q25_probit - .25).abs(),
    (results.q75_probit - .75).abs()], axis=1).max(axis=1) * 100
verdict = pd.DataFrame({"worst_interp_miss_pp": worst.round(2)})
verdict["reads"] = np.where(worst < 1.0, "excellent (<1pp)",
                    np.where(worst < 2.0, "good (<2pp)", "check — retrain this var"))
print("Worst |coverage - nominal| across interpolated q25/q75, probit scheme:\n")
verdict

Worst |coverage - nominal| across interpolated q25/q75, probit scheme:



,worst_interp_miss_pp,reads
var,,
tmax,1.7300,good (<2pp)
tmin,1.3100,good (<2pp)
precip,39.2700,check — retrain this var
wind,0.3700,excellent (<1pp)


## Reading it

- **Anchors** (`q10/q50/q90`) confirm the CQR shifts generalize out-of-sample — if these are
  off by >1pp the whole premise is shaky, not just the interpolation.
- **q25/q75 probit vs lin-τ**: probit should track nominal closely for tmax/tmin/wind; the
  gap between the two schemes is the cost of choosing wrong.
- **Conditional table**: marginal coverage can hide regime-dependent misses — if a column
  drifts far from nominal, interpolation is leaning on a shape assumption that breaks there
  (watch **precip**, whose zero-inflation is the known failure mode).
- **Verdict** `< ~1pp` ⇒ interpolate, don't retrain. A large precip miss ⇒ retrain precip only.